# <font color="#418FDE" size="6.5" uppercase>**Tree Regularization**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Explain CART construction and impurity reduction in practical terms. 
- Diagnose overfitting and instability in decision trees. 
- Apply pruning and structural constraints to improve generalization. 


## **1. Tree Mechanics**

### **1.1. CART Split Building**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_01_01.jpg?v=1784008202" width="250">



>* CART asks simple yes-or-no questions
>* Each split organizes outcomes step by step

>* CART tests many possible split questions
>* Best splits create more consistent child groups

>* Greedy splits shape each tree path
>* Too many splits can overfit noise



### **1.2. Impurity Reduction**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_01_02.jpg?v=1784008201" width="250">



>* Impure nodes mix different outcome classes
>* Good splits create clearer child groups

>* CART picks splits that most improve purity
>* Each choice is local, not globally optimal

>* Strong splits reveal useful data patterns
>* Too many splits can overfit noise



In [ ]:
#@title Python Code - Impurity Reduction

# This example shows impurity reduction in CART.
# We compare candidate splits on one feature.
# The best split gives the largest purity gain.

import numpy as np
import sklearn
from sklearn.tree import DecisionTreeClassifier

# Small labels represent loan default outcomes.
missed_payments = np.array([0, 0, 1, 1, 2, 2, 3, 3, 4, 4])
defaulted = np.array([0, 0, 0, 0, 1, 0, 1, 1, 1, 1])

# Gini impurity is lower when one class dominates.
def gini(labels):
    if len(labels) == 0:
        return 0.0
    class_zero_share = np.mean(labels == 0)
    class_one_share = np.mean(labels == 1)
    return 1.0 - class_zero_share ** 2 - class_one_share ** 2

# CART tests thresholds between sorted feature values.
thresholds = np.array([0.5, 1.5, 2.5, 3.5])
parent_gini = gini(defaulted)

# Compute weighted child impurity for each candidate split.
best_gain = -1.0
best_threshold = None

for threshold in thresholds:
    left_labels = defaulted[missed_payments <= threshold]
    right_labels = defaulted[missed_payments > threshold]
    left_weight = len(left_labels) / len(defaulted)
    right_weight = len(right_labels) / len(defaulted)
    child_gini = left_weight * gini(left_labels) + right_weight * gini(right_labels)
    gain = parent_gini - child_gini
    print(f"threshold <= {threshold}: impurity reduction = {gain:.3f}")
    if gain > best_gain:
        best_gain = gain
        best_threshold = threshold

# Fit one stump to confirm CART chooses the same split.
features = missed_payments.reshape(-1, 1)
stump = DecisionTreeClassifier(max_depth=1, random_state=42)
stump.fit(features, defaulted)

chosen_threshold = stump.tree_.threshold[0]
print(f"scikit-learn version: {sklearn.__version__}")
print(f"Best manual threshold: {best_threshold}")
print(f"CART stump threshold: {chosen_threshold:.1f}")



### **1.3. Tree Complexity**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_01_03.jpg?v=1784008204" width="250">



>* Deeper trees capture more detailed patterns
>* Too much detail can overfit training data

>* Complexity depends on tree size and shape
>* Tiny leaves can make predictions fragile

>* Early splits usually reduce impurity most
>* Limit complexity to avoid learning noise



## **2. Overfitting Control**

### **2.1. Tree Instability**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_02_01.jpg?v=1784008212" width="250">



>* Small data changes can reshape trees
>* Unstable splits may reflect sampling noise

>* Deep trees can learn random noise
>* Training accuracy may hide poor generalization

>* Compare trees across resampled datasets
>* Use regularization when predictions shift



In [ ]:
#@title Python Code - Tree Instability

# This example shows decision tree instability.
# Small data changes can alter early splits.
# Regularization makes the tree more stable.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier

# Create a small dataset with two similarly useful features.
features, target = make_classification(
    n_samples=240,
    n_features=4,
    n_informative=2,
    n_redundant=0,
    random_state=42,
)

# Validate the basic shape before fitting models.
if features.shape != (240, 4):
    raise ValueError("Expected 240 rows and 4 columns.")

# Train several trees on slightly different bootstrap samples.
rng = np.random.default_rng(42)
first_splits_deep = []
first_splits_regularized = []

for sample_number in range(12):
    sample_rows = rng.choice(len(target), size=len(target), replace=True)
    sample_features = features[sample_rows]
    sample_target = target[sample_rows]

    deep_tree = DecisionTreeClassifier(random_state=42)
    deep_tree.fit(sample_features, sample_target)
    first_splits_deep.append(deep_tree.tree_.feature[0])

    stable_tree = DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=20,
        random_state=42,
    )
    stable_tree.fit(sample_features, sample_target)
    first_splits_regularized.append(stable_tree.tree_.feature[0])

# Count how often each feature becomes the first split.
feature_names = np.array(["feature 0", "feature 1", "feature 2", "feature 3"])
deep_counts = np.bincount(first_splits_deep, minlength=4)
regularized_counts = np.bincount(first_splits_regularized, minlength=4)

print("scikit-learn version:", sklearn.__version__)
print("Deep tree first-split counts:", dict(zip(feature_names, deep_counts)))
print("Regularized first-split counts:", dict(zip(feature_names, regularized_counts)))

# Plot the first split choices for both tree settings.
x_positions = np.arange(len(feature_names))
bar_width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(x_positions - bar_width / 2, deep_counts, bar_width, label="Deep tree")
ax.bar(
    x_positions + bar_width / 2,
    regularized_counts,
    bar_width,
    label="Regularized tree",
)

ax.set_title("First Split Choices Across Slightly Different Samples")
ax.set_xlabel("Feature chosen at the root")
ax.set_ylabel("Number of bootstrap samples")
ax.set_xticks(x_positions)
ax.set_xticklabels(feature_names)
ax.legend()
plt.show()



### **2.2. Minimum Sample Limits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_02_02.jpg?v=1784008214" width="250">



>* Prevent splits based on too little evidence
>* Favor broader groups over noisy tiny patterns

>* Tiny leaves signal possible overfitting
>* Higher sample limits improve tree stability

>* Tune limits to data size and noise
>* Balance stability against missing useful patterns



In [ ]:
#@title Python Code - Minimum Sample Limits

# Compare tree sample limits on noisy data.
# Tiny leaves often memorize accidental training patterns.
# Larger leaves usually improve validation stability.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Create a small noisy classification dataset.
features, target = make_classification(
    n_samples=800,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    flip_y=0.18,
    class_sep=0.75,
    random_state=42,
)

# Split once so every tree faces the same test.
X_train, X_valid, y_train, y_valid = train_test_split(
    features,
    target,
    test_size=0.35,
    stratify=target,
    random_state=42,
)

# Validate the basic shape before modeling.
if X_train.shape[0] != y_train.shape[0]:
    raise ValueError("Training features and labels must match.")

# Try increasingly strict minimum leaf sizes.
leaf_limits = [1, 2, 5, 10, 20, 40, 80]
train_scores = []
valid_scores = []
leaf_counts = []

for leaf_limit in leaf_limits:
    tree = DecisionTreeClassifier(
        min_samples_leaf=leaf_limit,
        random_state=42,
    )
    tree.fit(X_train, y_train)
    train_scores.append(tree.score(X_train, y_train))
    valid_scores.append(tree.score(X_valid, y_valid))
    leaf_counts.append(tree.get_n_leaves())

# Print a compact summary of the overfitting pattern.
print("scikit-learn version:", sklearn.__version__)
print("min_leaf | train_acc | valid_acc | leaves")
for limit, train_score, valid_score, leaves in zip(
    leaf_limits,
    train_scores,
    valid_scores,
    leaf_counts,
):
    print(limit, "|", round(train_score, 3), "|", round(valid_score, 3), "|", leaves)

# Plot training and validation accuracy against leaf size.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(leaf_limits, train_scores, marker="o", label="Training accuracy")
ax.plot(leaf_limits, valid_scores, marker="o", label="Validation accuracy")
ax.set_title("Minimum Leaf Size Controls Tree Overfitting")

ax.set_xlabel("Minimum samples required in each leaf")
ax.set_ylabel("Accuracy")
ax.set_xscale("log")
ax.set_ylim(0.5, 1.02)
ax.legend()

plt.show()



### **2.3. Feature Constraints**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_02_03.jpg?v=1784008216" width="250">



>* Limit feature choices to reduce overfitting
>* Avoid noisy patterns that fail later

>* Constraints reduce tree sensitivity to data changes
>* Stable features improve interpretability and consistency

>* Remove unreliable features using domain knowledge
>* Keep enough signals for useful patterns



In [ ]:
#@title Python Code - Feature Constraints

# This example shows feature constraints in trees.
# We compare unrestricted and constrained feature choices.
# The result highlights overfitting and instability.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# A fixed seed makes the generated data repeatable.
X, y = make_classification(
    n_samples=800,
    n_features=20,
    n_informative=4,
    n_redundant=4,
    n_repeated=0,
    n_classes=2,
    flip_y=0.08,
    class_sep=0.8,
    random_state=42,
)

# This check keeps the lesson honest about the data shape.
if X.shape != (800, 20) or len(y) != 800:
    raise ValueError("Expected 800 rows and 20 features.")

# Stratification keeps both classes balanced in each split.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.35,
    stratify=y,
    random_state=42,
)

# The unrestricted tree may chase weak noisy features.
open_tree = DecisionTreeClassifier(random_state=42)
open_tree.fit(X_train, y_train)

# The constrained tree sees only a few features per split.
feature_limited_tree = DecisionTreeClassifier(
    max_features=4,
    random_state=42,
)
feature_limited_tree.fit(X_train, y_train)

# Accuracy gaps reveal overfitting on the training sample.
open_train = open_tree.score(X_train, y_train)
open_test = open_tree.score(X_test, y_test)
limited_train = feature_limited_tree.score(X_train, y_train)
limited_test = feature_limited_tree.score(X_test, y_test)

# Root features show how early choices can change structure.
open_root = open_tree.tree_.feature[0]
limited_root = feature_limited_tree.tree_.feature[0]

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Unrestricted tree train/test accuracy: {open_train:.3f} / {open_test:.3f}")
print(f"Feature-limited tree train/test accuracy: {limited_train:.3f} / {limited_test:.3f}")
print(f"Unrestricted tree depth and leaves: {open_tree.get_depth()} / {open_tree.get_n_leaves()}")
print(f"Feature-limited tree depth and leaves: {feature_limited_tree.get_depth()} / {feature_limited_tree.get_n_leaves()}")
print(f"Root feature index, unrestricted vs limited: {open_root} vs {limited_root}")



## **3. Pruning Guidance**

### **3.1. Cost Complexity Pruning**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_03_01.jpg?v=1784008206" width="250">



>* Prune branches with little added value
>* Favor stable patterns over memorized details

>* Balance training accuracy with tree simplicity
>* Remove fragile branches to improve generalization

>* Systematically compare simpler tree options
>* Improve stability, clarity, and unseen performance



In [ ]:
#@title Python Code - Cost Complexity Pruning

# This example shows cost complexity pruning.
# Pruning trades tree detail for generalization.
# The plot reveals a useful alpha choice.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Check that features and labels match.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match label count.")

# Split data so pruning is judged on unseen cases.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Grow one detailed tree to find pruning candidates.
base_tree = DecisionTreeClassifier(random_state=42)
path = base_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]

# Keep a small, readable set of alpha values.
alpha_positions = np.linspace(0, len(ccp_alphas) - 1, 12, dtype=int)
selected_alphas = ccp_alphas[alpha_positions]

# Train one tree for each pruning strength.
train_scores = []
test_scores = []
leaf_counts = []

for alpha in selected_alphas:
    tree = DecisionTreeClassifier(random_state=42, ccp_alpha=float(alpha))
    tree.fit(X_train, y_train)
    train_scores.append(tree.score(X_train, y_train))
    test_scores.append(tree.score(X_test, y_test))
    leaf_counts.append(tree.get_n_leaves())

# Identify the alpha with the best test accuracy.
best_index = int(np.argmax(test_scores))
best_alpha = selected_alphas[best_index]
best_test_score = test_scores[best_index]

print("scikit-learn version:", sklearn.__version__)
print("Unpruned leaves:", leaf_counts[0])
print("Best pruned leaves:", leaf_counts[best_index])
print("Best ccp_alpha:", round(float(best_alpha), 5))
print("Best test accuracy:", round(float(best_test_score), 3))

# Plot accuracy as pruning becomes stronger.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(leaf_counts, train_scores, marker="o", label="Training accuracy")
ax.plot(leaf_counts, test_scores, marker="o", label="Test accuracy")

# Smaller leaf counts mean stronger pruning.
ax.invert_xaxis()
ax.set_title("Cost Complexity Pruning Path")
ax.set_xlabel("Number of leaves after pruning")
ax.set_ylabel("Accuracy")
ax.legend()
plt.show()



### **3.2. Choosing Alpha**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_03_02.jpg?v=1784008210" width="250">



>* Alpha controls the tree complexity penalty
>* Pruning removes fragile, overly specific branches

>* Test alpha values on unseen data
>* Match metrics to task and error costs

>* Prefer simpler trees when scores are similar
>* Balance accuracy, clarity, stability, and consequences



In [ ]:
#@title Python Code - Choosing Alpha

# This example chooses pruning strength for a tree.
# Alpha penalizes branches that add little validation value.
# The plot reveals a simpler competitive tree.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Check that features and labels match.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match label count.")

# Split once so validation data stays unseen.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Grow one tree to discover possible pruning alphas.
base_tree = DecisionTreeClassifier(random_state=42)
path = base_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]

# Keep a small, readable set of candidate alphas.
alpha_indexes = np.linspace(0, len(ccp_alphas) - 1, 12, dtype=int)
candidate_alphas = np.unique(ccp_alphas[alpha_indexes])

train_scores = []
valid_scores = []
leaf_counts = []

# Fit one pruned tree for each candidate alpha.
for alpha in candidate_alphas:
    tree = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    tree.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, tree.predict(X_train)))
    valid_scores.append(accuracy_score(y_valid, tree.predict(X_valid)))
    leaf_counts.append(tree.get_n_leaves())

best_score = max(valid_scores)
best_positions = np.where(np.array(valid_scores) == best_score)[0]
best_position = best_positions[-1]
chosen_alpha = candidate_alphas[best_position]
chosen_leaves = leaf_counts[best_position]

print("scikit-learn version:", sklearn.__version__)
print("Best validation accuracy:", round(best_score, 3))
print("Chosen alpha:", round(float(chosen_alpha), 5))
print("Leaves in chosen tree:", chosen_leaves)

# Plot accuracy against alpha to show the tradeoff.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(candidate_alphas, train_scores, marker="o", label="training")
ax.plot(candidate_alphas, valid_scores, marker="o", label="validation")

ax.axvline(chosen_alpha, color="gray", linestyle="--", label="chosen alpha")
ax.set_title("Choosing Alpha for Cost Complexity Pruning")
ax.set_xlabel("ccp_alpha penalty")
ax.set_ylabel("accuracy")
ax.legend()

plt.show()



### **3.3. Monotonic Constraints**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_16/Lecture_B/image_03_03.jpg?v=1784008208" width="250">



>* Encode expected prediction direction from domain knowledge
>* Prevent noisy reversals without forcing straight lines

>* Limit noisy patterns to improve generalization
>* Keep predictions stable and domain-aligned

>* Constrain only clearly reliable feature directions
>* Combine constraints with broader regularization



In [ ]:
#@title Python Code - Monotonic Constraints

# This example shows monotonic tree constraints.
# Constraints prevent implausible prediction reversals.
# The plot compares constrained and unconstrained trees.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.tree import DecisionTreeRegressor

# A fixed generator makes the noisy example repeatable.
rng = np.random.default_rng(42)

# The true relationship increases as the feature increases.
x = np.linspace(0, 10, 80)
noise = rng.normal(0, 1.2, size=x.shape)
y = 2.0 * x + noise

# A few unusual points can tempt a tree into reversals.
y[55:62] = y[55:62] - 10
X = x.reshape(-1, 1)

# Validate the simple one-feature training data.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature and target lengths must match.")

# Fit one flexible tree and one monotonic tree.
free_tree = DecisionTreeRegressor(max_depth=4, random_state=42)
mono_tree = DecisionTreeRegressor(
    max_depth=4,
    random_state=42,
    monotonic_cst=[1],
)
free_tree.fit(X, y)
mono_tree.fit(X, y)

# Predict on a smooth grid to inspect prediction direction.
grid = np.linspace(0, 10, 200).reshape(-1, 1)
free_pred = free_tree.predict(grid)
mono_pred = mono_tree.predict(grid)

# Count downward steps as a simple reversal diagnostic.
free_reversals = int(np.sum(np.diff(free_pred) < -1e-9))
mono_reversals = int(np.sum(np.diff(mono_pred) < -1e-9))

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Unconstrained tree downward steps: {free_reversals}")
print(f"Monotonic tree downward steps: {mono_reversals}")

# Plot the data and both fitted prediction curves.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, s=25, alpha=0.65, label="noisy training data")
ax.plot(grid.ravel(), free_pred, label="unconstrained tree")
ax.plot(grid.ravel(), mono_pred, label="monotonic increasing tree")
ax.set_title("Monotonic constraint as a regularization guardrail")
ax.set_xlabel("Feature value")
ax.set_ylabel("Predicted target")
ax.legend()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Tree Regularization**</font>


In this lecture, you learned to:
- Explain CART construction and impurity reduction in practical terms. 
- Diagnose overfitting and instability in decision trees. 
- Apply pruning and structural constraints to improve generalization. 

In the next Module (Module 17), we will go over 'Forests Bagging'